In [8]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [9]:
# Helper functions
from anthropic.types import Message

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])



def _trunc(s, n=40):
    s = str(s)
    return s if len(s) <= n else f"{s[:n]}...[{len(s)} chars]"


def show(message, width=100):
    """Print a Message in a readable form (signatures/data truncated)."""
    print(f"model={message.model}  stop_reason={message.stop_reason}")
    u = message.usage
    print(
        f"tokens: in={u.input_tokens} out={u.output_tokens}"
        + (
            f" (thinking={u.output_tokens_details.thinking_tokens})"
            if getattr(u, "output_tokens_details", None)
            else ""
        )
        + f" cache_read={u.cache_read_input_tokens} cache_write={u.cache_creation_input_tokens}"
    )

    for i, block in enumerate(message.content):
        print(f"\n{'-' * width}\n[{i}] {block.type}")
        if block.type == "thinking":
            print(f"    signature: {_trunc(block.signature)}")
            print(block.thinking)
        elif block.type == "redacted_thinking":
            print(f"    data: {_trunc(block.data)}")
        elif block.type == "text":
            print(block.text)
        elif block.type == "tool_use":
            print(f"    name: {block.name}\n    input: {block.input}")
        else:
            print(block)

In [10]:
messages = []

# add_user_message(messages, thinking_test_str)
add_user_message(messages, "what is the purpose of life?")

show(chat(messages, thinking=True))

model=claude-sonnet-4-5-20250929  stop_reason=end_turn
tokens: in=43 out=428 (thinking=175) cache_read=0 cache_write=0

----------------------------------------------------------------------------------------------------
[0] thinking
    signature: EuEICpQBCA8YAipAPxYtJZXoLkfmu8xxVLdADXd/...[1504 chars]
This is one of the most profound philosophical questions humans ask. There's no single universally accepted answer, as it depends on philosophical, religious, cultural, and personal perspectives. Let me provide a thoughtful, balanced response that acknowledges different viewpoints:

1. Religious/spiritual perspectives offer various answers (serving God, achieving enlightenment, etc.)
2. Philosophical perspectives range from existentialism (we create our own meaning) to nihilism (there is no inherent purpose) to various other frameworks
3. Scientific/biological perspective might say survival and reproduction
4. Humanistic perspectives often focus on growth, connection, contribution, and 